# 04 · Capa Semántica — Metric Views

**Objetivo:** crear una **capa semántica** sobre el modelo estrella gold usando
**Metric Views** de Databricks. Las metric views definen métricas de negocio
(KPIs) de forma centralizada y reutilizable — cualquier dashboard, query o
herramienta BI consume las mismas definiciones, eliminando discrepancias.

### ¿Qué es una Metric View?
Una vista especial de Unity Catalog que declara en **YAML**:
* **`measures`** — expresiones agregadas (`SUM`, `AVG`, `COUNT`) con nombre de negocio
* **`dimensions`** — ejes de análisis para hacer *slice & dice*
* **`joins`** — relaciones con tablas de dimensión (star schema)
* **`filter`** — restricciones predefinidas

```sql
CREATE OR REPLACE VIEW catalog.schema.obras_kpis
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: catalog.schema.gold_fact_obras
  joins:
    - name: dim_estados
      source: catalog.schema.gold_dim_estados
      on: source.IdEstadoRep = dim_estados.IdEstadoRep
      rely:
        at_most_one_match: true
  dimensions:
    - name: estado
      expr: dim_estados.NombreEstado
      comment: Estado de la república
  measures:
    - name: total_importe_contrato
      expr: SUM(source.ImporteContratoMxn)
      comment: Inversión total en MXN
      format:
        type: currency
        currency_code: MXN
        decimal_places:
          type: exact
          places: 2
$$
```

### Para consultar una metric view
Las métricas se envuelven con `MEASURE()` y siempre se usa `GROUP BY ALL`:
```sql
SELECT `estado`, MEASURE(`total_importe_contrato`) AS importe
FROM catalog.schema.obras_kpis
GROUP BY ALL
```

### Ventajas
* **Una sola fuente de verdad** para los KPIs
* Los dashboards consumen la metric view con `MEASURE()` en vez de SQL ad-hoc
* Cambiar la lógica de una métrica se refleja en todos los consumidores
* Metadata semántica: `comment`, `display_name`, `format`, `synonyms`

> Requiere haber ejecutado la **pipeline del lab 03** (tablas gold).

In [0]:
%run ./_setup

## 1. Métricas de negocio propuestas

Basadas en la tabla de hechos `gold_fact_obras` (318 obras de construcción):

### Métricas financieras
| Métrica | Expresión | Descripción |
|---|---|---|
| `total_importe_contrato` | `SUM(ImporteContratoMxn)` | Inversión total en MXN |
| `importe_promedio_obra` | `AVG(ImporteContratoMxn)` | Importe promedio por obra |
| `total_icm` | `SUM(ICM)` | Importe de construcción y montaje total |
| `total_cptt` | `SUM(CPTT)` | Costo total del proyecto |

### Métricas de avance
| Métrica | Expresión | Descripción |
|---|---|---|
| `avance_promedio_construccion` | `AVG(AvanceActualConstruccion)` | % avance promedio de construcción |
| `avance_promedio_general` | `AVG(AvanceGeneralAP)` | % avance general promedio |
| `obras_completadas` | `COUNT_IF(AvanceActualConstruccion >= 100)` | Obras con avance al 100% |
| `obras_en_riesgo` | `COUNT_IF(AvanceActualConstruccion < 50)` | Obras con menos de 50% de avance |

### Métricas de volumen
| Métrica | Expresión | Descripción |
|---|---|---|
| `total_obras` | `COUNT(*)` | Número total de obras |
| `total_kilometros` | `SUM(Kilometros)` | Km de líneas de transmisión |
| `total_mva` | `SUM(MVA)` | Capacidad total en MVA (subestaciones) |

## 2. Dimensiones para análisis

| Dimensión | Fuente | Ejemplo de pregunta de negocio |
|---|---|---|
| `estado` | `gold_dim_estados.NombreEstado` | ¿Cuánto se invierte por estado? |
| `tipo_obra` | `gold_dim_tipos_obra.TipoObra` | Línea vs subestación: ¿dónde hay más inversión? |
| `proyecto` | `gold_dim_proyectos.Nombre` | ¿Qué proyecto tiene más obras? |
| `residencia_regional` | `gold_dim_residencias.Nombre` | Inversión por residencia regional |
| `estado_proyecto` | `gold_dim_estados_proyecto.Nombre` | Obras en ejecución vs terminadas |
| `voltaje` | `gold_fact_obras.kV` | Distribución de obras por nivel de voltaje |
| `situacion_actual` | `gold_fact_obras.SituacionActual` | Obras suspendidas vs en construcción |

## 3. Actividad: genera tus metric views con Genie Code

### Pasos:
1. **Ejecuta la celda de abajo** para obtener el prompt con tus variables.
2. **Copia el prompt** y pégalo en **Genie Code** (`Cmd+I` / `Ctrl+I`).
3. Revisa que el SQL generado incluya:
   * `CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML AS $$ ... $$`
   * `version: 1.1` en el YAML
   * Bloques `joins`, `dimensions` y `measures` con `comment` y `format`
   * `rely: at_most_one_match: true` en cada join
4. **Ejecuta** el SQL generado para crear las metric views.

### Para consumir la metric view:
**Reglas clave:**
* Las métricas se envuelven con `MEASURE()`
* Siempre usar `GROUP BY ALL`
* Nunca usar `SELECT *`
* Los nombres van entre backticks

```sql
-- Importe total por estado
SELECT `estado`, MEASURE(`total_importe_contrato`) AS importe
FROM dipidev.default.obras_kpis
GROUP BY ALL;

-- Avance promedio por tipo de obra
SELECT `tipo_obra`,
       MEASURE(`avance_promedio_construccion`) AS avance,
       MEASURE(`total_obras`) AS obras
FROM dipidev.default.obras_kpis
GROUP BY ALL;

-- Total general (sin dimensiones)
SELECT MEASURE(`total_importe_contrato`) AS inversion_total,
       MEASURE(`total_obras`) AS obras
FROM dipidev.default.obras_kpis
GROUP BY ALL;
```

In [0]:
prompt = f"""
Genera metric views para la capa semántica del proyecto CFE.
Catálogo: {CATALOG} | Esquema: {WORK_SCHEMA}

CREATE OR REPLACE VIEW catalog.schema.nombre
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: catalog.schema.tabla
  joins: [...]      # relaciones con dimensiones
  dimensions: [...] # ejes de análisis (name + expr + comment)
  measures: [...]   # agregaciones (name + expr + comment + format)
  filter: ...       # filtro base opcional
$$

Para consultar: SELECT `dim`, MEASURE(`metrica`) FROM vista GROUP BY ALL

===== TABLAS DISPONIBLES (modelo estrella gold) =====

Hecho: {CATALOG}.{WORK_SCHEMA}.gold_fact_obras (318 filas)
  Columnas clave:
  - IdObra (PK), IdProyecto (FK), IdTipoObra (FK), IdEstadoRep (FK),
    IdResidenciaRegional (FK), IdEstadoProyecto (FK)
  - ImporteContratoMxn (double), ICM (double), CPTT (double), CN (double), ACB (double)
  - AvanceActualConstruccion (double 0-100), AvanceGeneralAP (double 0-100)
  - ActividadesPrevias (double)
  - FechaInicioProgramada (date), FechaTerminacionProgramada (date), FechaInicioReal (date)
  - kV (string), NombreObra (string), SituacionActual (string)
  - Kilometros (double, NULL si no es línea), MVA (double, NULL si no es subestación)
  - Circuitos (long), MVAr (double), Alimentadores (long)

Dimensiones:
  - {CATALOG}.{WORK_SCHEMA}.gold_dim_proyectos (78 filas)
    IdProyecto (PK), Nombre, Alcance, Cliente, Contratista, MontoPMC, Latitud, Longitud
  - {CATALOG}.{WORK_SCHEMA}.gold_dim_estados (32 filas)
    IdEstadoRep (PK), NombreEstado, Abreviatura
  - {CATALOG}.{WORK_SCHEMA}.gold_dim_tipos_obra (2 filas)
    IdTipoObra (PK), TipoObra
  - {CATALOG}.{WORK_SCHEMA}.gold_dim_residencias (7 filas)
    IdResidenciaRegional (PK), Nombre, Siglas
  - {CATALOG}.{WORK_SCHEMA}.gold_dim_estados_proyecto (6 filas)
    IdEstadoProyecto (PK), Nombre

===== METRIC VIEW: obras_kpis =====

CREATE OR REPLACE VIEW {CATALOG}.{WORK_SCHEMA}.obras_kpis
WITH METRICS LANGUAGE YAML AS $$ ... $$

version: 1.1
source: {CATALOG}.{WORK_SCHEMA}.gold_fact_obras

joins (many_to_one con rely at_most_one_match: true en todos):
  - dim_estados:          on source.IdEstadoRep = dim_estados.IdEstadoRep
  - dim_tipos_obra:       on source.IdTipoObra = dim_tipos_obra.IdTipoObra
  - dim_proyectos:        on source.IdProyecto = dim_proyectos.IdProyecto
  - dim_residencias:      on source.IdResidenciaRegional = dim_residencias.IdResidenciaRegional
  - dim_estados_proyecto: on source.IdEstadoProyecto = dim_estados_proyecto.IdEstadoProyecto

dimensions (cada una con name, expr y comment):
  - estado            : dim_estados.NombreEstado      ("Estado de la república")
  - tipo_obra         : dim_tipos_obra.TipoObra       ("Línea o Subestación")
  - proyecto          : dim_proyectos.Nombre           ("Nombre del proyecto")
  - residencia        : dim_residencias.Nombre         ("Residencia regional CFE")
  - estado_proyecto   : dim_estados_proyecto.Nombre    ("Estado del proyecto: ejecución, terminado, etc.")
  - voltaje           : source.kV                      ("Nivel de voltaje de la obra")
  - situacion_actual  : source.SituacionActual         ("Situación actual de la obra")
  - cliente           : dim_proyectos.Cliente          ("Cliente del proyecto")
  - contratista       : dim_proyectos.Contratista      ("Contratista asignado")
  - nombre_obra       : source.NombreObra              ("Nombre de la obra de construcción")

measures (cada una con name, expr, comment y format cuando aplique):
  - total_importe_contrato:       SUM(source.ImporteContratoMxn)
    format: currency, MXN, 2 decimales
    comment: "Inversión total en contratos MXN"
    synonyms: [inversión, monto total]
  - importe_promedio_obra:        SUM(source.ImporteContratoMxn) / COUNT(1)
    format: currency, MXN, 2 decimales
    comment: "Importe promedio por obra"
  - total_icm:                    SUM(source.ICM)
    format: currency, MXN, 2 decimales
    comment: "Importe de construcción y montaje total"
  - total_cptt:                   SUM(source.CPTT)
    format: currency, MXN, 2 decimales
    comment: "Costo programado total de trabajo"
  - avance_promedio_construccion: AVG(source.AvanceActualConstruccion)
    format: number, 2 decimales
    comment: "Porcentaje promedio de avance en construcción"
  - avance_promedio_general:      AVG(source.AvanceGeneralAP)
    format: number, 2 decimales
    comment: "Porcentaje promedio de avance general"
  - obras_completadas:            COUNT(1) FILTER (WHERE source.AvanceActualConstruccion >= 100)
    comment: "Número de obras con avance al 100%"
  - obras_en_riesgo:              COUNT(1) FILTER (WHERE source.AvanceActualConstruccion < 50)
    comment: "Número de obras con menos de 50% de avance"
  - total_obras:                  COUNT(1)
    comment: "Número total de obras"
    synonyms: [cantidad de obras, número de obras]
  - total_kilometros:             SUM(source.Kilometros)
    format: number, 2 decimales
    comment: "Kilómetros de líneas de transmisión"
  - total_mva:                    SUM(source.MVA)
    format: number, 2 decimales
    comment: "Capacidad total en MVA (subestaciones)"

===== METRIC VIEW 2: proyectos_kpis =====

CREATE OR REPLACE VIEW {CATALOG}.{WORK_SCHEMA}.proyectos_kpis
WITH METRICS LANGUAGE YAML AS $$ ... $$

version: 1.1
source: >
  SELECT p.*, f.ImporteContratoMxn, f.AvanceActualConstruccion, f.IdObra
  FROM {CATALOG}.{WORK_SCHEMA}.gold_dim_proyectos p
  LEFT JOIN {CATALOG}.{WORK_SCHEMA}.gold_fact_obras f ON p.IdProyecto = f.IdProyecto
comment: "KPIs a nivel de proyecto de construcción CFE"

dimensions:
  - nombre_proyecto: source.Nombre     ("Nombre del proyecto")
  - alcance:         source.Alcance    ("Alcance del proyecto")
  - cliente:         source.Cliente    ("Cliente del proyecto")
  - contratista:     source.Contratista ("Contratista asignado")

measures:
  - total_obras_proyecto:     COUNT(source.IdObra)
    comment: "Número de obras en el proyecto"
  - importe_total_proyecto:   SUM(source.ImporteContratoMxn)
    format: currency, MXN, 2 decimales
    comment: "Inversión total del proyecto"
  - avance_promedio_proyecto: AVG(source.AvanceActualConstruccion)
    format: number, 2 decimales
    comment: "Avance promedio de construcción del proyecto"

===== NOTAS TÉCNICAS =====
1. Usar `version: 1.1` en el YAML (no 1 solo, da error)
2. Todas las expresiones de measures deben referenciar `source.columna` o `join_name.columna`
3. Cada join necesita `rely: at_most_one_match: true` (son many_to_one confirmados)
4. Usar `comment` y `format` en cada medida para metadata semántica
5. Las medidas se consultan con MEASURE(`nombre`), siempre con GROUP BY ALL
6. NO se puede usar SELECT * en metric views
"""

print(prompt)

In [0]:
%sql
CREATE OR REPLACE VIEW dipidev.default.obras_kpis
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: dipidev.default.gold_fact_obras
comment: KPIs de obras de construcción CFE — inversión, avance y volumen
joins:
  - name: dim_estados
    source: dipidev.default.gold_dim_estados
    on: source.IdEstadoRep = dim_estados.IdEstadoRep
    rely:
      at_most_one_match: true
  - name: dim_tipos_obra
    source: dipidev.default.gold_dim_tipos_obra
    on: source.IdTipoObra = dim_tipos_obra.IdTipoObra
    rely:
      at_most_one_match: true
  - name: dim_proyectos
    source: dipidev.default.gold_dim_proyectos
    on: source.IdProyecto = dim_proyectos.IdProyecto
    rely:
      at_most_one_match: true
  - name: dim_residencias
    source: dipidev.default.gold_dim_residencias
    on: source.IdResidenciaRegional = dim_residencias.IdResidenciaRegional
    rely:
      at_most_one_match: true
  - name: dim_estados_proyecto
    source: dipidev.default.gold_dim_estados_proyecto
    on: source.IdEstadoProyecto = dim_estados_proyecto.IdEstadoProyecto
    rely:
      at_most_one_match: true
dimensions:
  - name: estado
    expr: dim_estados.NombreEstado
    comment: Estado de la república
    display_name: Estado
  - name: tipo_obra
    expr: dim_tipos_obra.TipoObra
    comment: Línea o Subestación
    display_name: Tipo de Obra
  - name: proyecto
    expr: dim_proyectos.Nombre
    comment: Nombre del proyecto
    display_name: Proyecto
  - name: residencia
    expr: dim_residencias.Nombre
    comment: Residencia regional CFE
    display_name: Residencia Regional
  - name: estado_proyecto
    expr: dim_estados_proyecto.Nombre
    comment: "Estado del proyecto: ejecución, terminado, etc."
    display_name: Estado del Proyecto
  - name: voltaje
    expr: source.kV
    comment: Nivel de voltaje de la obra
    display_name: Voltaje (kV)
  - name: situacion_actual
    expr: source.SituacionActual
    comment: Situación actual de la obra
    display_name: Situación Actual
  - name: cliente
    expr: dim_proyectos.Cliente
    comment: Cliente del proyecto
    display_name: Cliente
  - name: contratista
    expr: dim_proyectos.Contratista
    comment: Contratista asignado
    display_name: Contratista
  - name: nombre_obra
    expr: source.NombreObra
    comment: Nombre de la obra de construcción
    display_name: Nombre de Obra
measures:
  - name: total_importe_contrato
    expr: SUM(source.ImporteContratoMxn)
    comment: Inversión total en contratos MXN
    display_name: Importe Total Contrato
    synonyms:
      - inversión
      - monto total
    format:
      type: currency
      currency_code: MXN
      decimal_places:
        type: exact
        places: 2
  - name: importe_promedio_obra
    expr: SUM(source.ImporteContratoMxn) / COUNT(1)
    comment: Importe promedio por obra
    display_name: Importe Promedio por Obra
    format:
      type: currency
      currency_code: MXN
      decimal_places:
        type: exact
        places: 2
  - name: total_icm
    expr: SUM(source.ICM)
    comment: Importe de construcción y montaje total
    display_name: ICM Total
    format:
      type: currency
      currency_code: MXN
      decimal_places:
        type: exact
        places: 2
  - name: total_cptt
    expr: SUM(source.CPTT)
    comment: Costo programado total de trabajo
    display_name: CPTT Total
    format:
      type: currency
      currency_code: MXN
      decimal_places:
        type: exact
        places: 2
  - name: avance_promedio_construccion
    expr: AVG(source.AvanceActualConstruccion)
    comment: Porcentaje promedio de avance en construcción
    display_name: Avance Promedio Construcción
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: avance_promedio_general
    expr: AVG(source.AvanceGeneralAP)
    comment: Porcentaje promedio de avance general
    display_name: Avance Promedio General
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: obras_completadas
    expr: COUNT(1) FILTER (WHERE source.AvanceActualConstruccion >= 100)
    comment: Número de obras con avance al 100%
    display_name: Obras Completadas
  - name: obras_en_riesgo
    expr: COUNT(1) FILTER (WHERE source.AvanceActualConstruccion < 50)
    comment: Número de obras con menos de 50% de avance
    display_name: Obras en Riesgo
  - name: total_obras
    expr: COUNT(1)
    comment: Número total de obras
    display_name: Total Obras
    synonyms:
      - cantidad de obras
      - número de obras
  - name: total_kilometros
    expr: SUM(source.Kilometros)
    comment: Kilómetros de líneas de transmisión
    display_name: Total Kilómetros
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: total_mva
    expr: SUM(source.MVA)
    comment: Capacidad total en MVA (subestaciones)
    display_name: Total MVA
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
$$

In [0]:
%sql
CREATE OR REPLACE VIEW dipidev.default.proyectos_kpis
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: >
  SELECT p.IdProyecto, p.Nombre, p.Alcance, p.Cliente, p.Contratista,
         p.MontoPMC, p.Latitud, p.Longitud,
         f.ImporteContratoMxn, f.AvanceActualConstruccion, f.IdObra
  FROM dipidev.default.gold_dim_proyectos p
  LEFT JOIN dipidev.default.gold_fact_obras f ON p.IdProyecto = f.IdProyecto
comment: KPIs a nivel de proyecto de construcción CFE
dimensions:
  - name: nombre_proyecto
    expr: source.Nombre
    comment: Nombre del proyecto
    display_name: Proyecto
  - name: alcance
    expr: source.Alcance
    comment: Alcance del proyecto
    display_name: Alcance
  - name: cliente
    expr: source.Cliente
    comment: Cliente del proyecto
    display_name: Cliente
  - name: contratista
    expr: source.Contratista
    comment: Contratista asignado
    display_name: Contratista
measures:
  - name: total_obras_proyecto
    expr: COUNT(source.IdObra)
    comment: Número de obras en el proyecto
    display_name: Obras en Proyecto
  - name: importe_total_proyecto
    expr: SUM(source.ImporteContratoMxn)
    comment: Inversión total del proyecto
    display_name: Inversión Total
    format:
      type: currency
      currency_code: MXN
      decimal_places:
        type: exact
        places: 2
  - name: avance_promedio_proyecto
    expr: AVG(source.AvanceActualConstruccion)
    comment: Avance promedio de construcción del proyecto
    display_name: Avance Promedio
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
$$

## 4. Verificar las metric views

Después de ejecutar el SQL generado, prueba las metric views con estas consultas:

In [0]:
%sql
-- ¿Cuánto se invierte por tipo de obra?
SELECT `tipo_obra`,
       MEASURE(`total_importe_contrato`) AS importe,
       MEASURE(`total_obras`) AS obras,
       MEASURE(`avance_promedio_construccion`) AS avance_prom
FROM dipidev.default.obras_kpis
GROUP BY ALL

In [0]:
%sql
-- Top 10 proyectos por inversión
WITH ranked AS (
  SELECT `proyecto`,
         MEASURE(`total_importe_contrato`) AS importe,
         MEASURE(`total_obras`) AS obras,
         MEASURE(`avance_promedio_construccion`) AS avance_prom,
         ROW_NUMBER() OVER (ORDER BY MEASURE(`total_importe_contrato`) DESC) AS rn
  FROM dipidev.default.obras_kpis
  GROUP BY ALL
)
SELECT `proyecto`, importe, obras, avance_prom
FROM ranked
WHERE rn <= 10